# Second-order plant, then PID feedback
Run the two code cells in order, then move the sliders. Plots update when you release a slider. Plant sliders in the first section also update the PID comparison below. If you rerun the first cell, rerun the second to reconnect it.

Requires NumPy, SciPy, pandas, tqdm, Matplotlib, and **ipywidgets** in the selected kernel. If widgets are missing, run `%pip install ipywidgets` in a temporary notebook cell, then restart the kernel.

We use a stable plant with optional dead time: $G(s)=K\omega_n^2/(s^2+2\zeta\omega_n s+\omega_n^2)e^{-Ls}$. For a series RLC circuit with capacitor-voltage output, $K=1$, $\omega_n=1/\sqrt{LC}$, and $\zeta=(R/2)\sqrt{C/L}$. All responses start from rest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
project_root = next((root for root in (Path.cwd(), Path.cwd().parent)
                     if (root / 'src' / 'data_gen.py').exists()), None)
if project_root is None:
    raise RuntimeError('Open this notebook from the project root or simulator folder.')
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))
from data_gen import Plant, PIDGains, PlantSimulator
simulator = PlantSimulator()
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO, StringIO
from contextlib import redirect_stdout
from html import escape

# Disconnect callbacks explicitly; closing an Output alone leaves observers attached.
for binding in globals().get('_plot_bindings', {}).values():
    for control in binding['controls'].values():
        control.unobserve(binding['callback'], names='value')
    binding['output'].close()
_plot_bindings = {}

def disconnect_plot(name):
    binding = _plot_bindings.pop(name, None)
    if binding is not None:
        for control in binding['controls'].values():
            control.unobserve(binding['callback'], names='value')
        binding['output'].close()

def bind_plot(name, function, controls):
    disconnect_plot(name)
    image = widgets.Image(format='png', layout=widgets.Layout(max_width='100%'))
    details = widgets.HTML()
    output = widgets.VBox([image, details])
    def redraw(change=None):
        # Replace bytes in one existing image; never append notebook plot outputs.
        text = StringIO()
        with redirect_stdout(text), plt.ioff():
            fig = function(**{key: control.value for key, control in controls.items()})
            buffer = BytesIO()
            try:
                fig.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
                image.value = buffer.getvalue()
            finally:
                plt.close(fig)
                buffer.close()
        details.value = '<pre>' + escape(text.getvalue()) + '</pre>'
    for control in controls.values():
        control.observe(redraw, names='value')
    _plot_bindings[name] = {
        'controls': controls, 'callback': redraw, 'output': output,
    }
    redraw()
    return output

def slider(label, value, lower, upper, step):
    return widgets.FloatSlider(
        description=label, value=value, min=lower, max=upper, step=step,
        continuous_update=False, style={'description_width': '150px'},
        layout=widgets.Layout(width='550px'), readout_format='.2f')

plant_controls = {
    'zeta': slider('Damping zeta', 0.2, 0.05, 2.0, 0.05),
    'wn': slider('Frequency wn (rad/s)', 1.0, 0.2, 10.0, 0.1),
    'K': slider('Plant gain K', 1.0, 0.1, 3.0, 0.1),
    'delay': slider('Dead time (s)', 0.2, 0.0, 1.0, 0.01),
    'setpoint': slider('Setpoint', 1.0, 0.1, 5.0, 0.1),
    'duration': slider('Duration (s)', 30.0, 5.0, 100.0, 1.0),
}

def plot_plant(zeta, wn, K, delay, setpoint, duration):
    assert zeta > 0 and wn > 0 and K > 0 and duration > 0
    assert setpoint > 0, 'Use a positive setpoint for this experiment.'
    t = np.linspace(0, duration, 3000)
    
    plant = Plant(zeta, wn, delay, K)
    _, unit_response = simulator.open_loop(plant, t)
    y_open = setpoint * unit_response
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, y_open, label='Plant alone: constant input = setpoint')
    ax.axhline(setpoint, color='black', linestyle='--', label='Desired output')
    ax.set(title='Open-loop step response', xlabel='Time (s)', ylabel='Output y(t)')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.close(fig)  # Prevent automatic inline display from duplicating the figure.

    print(f'Open-loop final output approaches K Ã— input = {K * setpoint:g}.')
    print('No controller is present: the input stays constant regardless of the output.')
    return fig

plant_output = bind_plot('plant', plot_plant, plant_controls)
display(widgets.VBox(list(plant_controls.values())), plant_output)


## Add unity negative ideal PID feedback
The error is $e=r-y$, with $C(s)=K_p+K_i/s+K_ds$. The output transfer function is $CG/(1+CG)$.
The notebook imports the generator’s plant/controller model, including second-order Padé delay. Open-loop output uses the analytical true-delay response.
An ideal derivative on a reference step produces an input impulse of weight $rK_d$ at time zero. Only the regular part of the control input is plotted. No actuator limits, noise, or automatic tuning are included.
Use the sliders to compare P, PI, PD, and PID control. Metrics describe the displayed window, not the generator’s independent validation.


In [ ]:
# Run the plant cell first. Its sliders also update these feedback plots.
disconnect_plot('pid')
pid_controls = {
    'Kp': slider('Kp', 1.0, 0.0, 20.0, 0.1),
    'Ki': slider('Ki', 0.5, 0.0, 20.0, 0.1),
    'Kd': slider('Kd', 0.5, 0.0, 10.0, 0.1),
}

def plot_feedback(zeta, wn, K, delay, setpoint, duration, Kp, Ki, Kd):
    t = np.linspace(0, duration, 5000)
    plant = Plant(zeta, wn, delay, K)
    gains = PIDGains(Kp, Ki, Kd)
    _, y_open = simulator.open_loop(plant, t)
    _, y_pid = simulator.close_loop(plant, gains, t)
    u_pid, impulse_weight = simulator.control_effort(plant, gains, t)
    y_open *= setpoint
    y_pid *= setpoint
    u_pid *= setpoint
    _, closed_den = simulator.closed_loop_coefficients(plant, gains)
    poles = np.roots(closed_den)
    stable = bool(np.all(poles.real < 0))
    print(f'Ideal derivative input impulse at t=0: weight {setpoint * impulse_weight:g} (not plotted).')
    print('Closed-loop poles:', poles)
    print('Second-order Pade model asymptotically stable:', stable)
    
    fig, (ax_y, ax_u) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    ax_y.plot(t, y_open, label='Plant alone')
    ax_y.axhline(setpoint, color='black', linestyle='--', label='Setpoint')
    if stable and np.all(np.isfinite(y_pid)) and np.all(np.isfinite(u_pid)):
        ax_y.plot(t, y_pid, label='Plant with PID feedback')
        ax_u.plot(t, u_pid, color='tab:purple', label='Regular PID input (impulse omitted)')
        overshoot = max(0.0, (np.max(y_pid) / setpoint - 1) * 100)
        print(f'Sampled overshoot: {overshoot:.2f}%')
        print(f'Error at end of simulation: {setpoint - y_pid[-1]:.5g}')
    else:
        print('PID traces hidden: unstable/marginal or nonfinite response. Adjust gains.')
    ax_u.axhline(setpoint, color='gray', linestyle=':', label='Constant input without PID')
    ax_y.set(title='Open loop versus PID feedback', ylabel='Output y(t)')
    ax_u.set(xlabel='Time (s)', ylabel='Regular control input u(t)')
    for ax in (ax_y, ax_u):
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.close(fig)  # Display exactly once, inside this plot output.

    print('End error is measured at the final time, not necessarily at steady state.')
    return fig

pid_output = bind_plot('pid', plot_feedback, {**plant_controls, **pid_controls})
display(widgets.VBox(list(pid_controls.values())), pid_output)
